# SW07 Posterior Preflight on Colab GPU

This notebook runs the same SW07 posterior likelihood and gradient preflight used on GPUHub. It is meant to answer whether a slow HMC run is caused by sampling or by the compiled log-density/gradient itself.

Use `Runtime -> Change runtime type -> T4 GPU` before running. The cell clones the Python repo, installs a CUDA JAX wheel selected from `nvidia-smi`, runs `posterior_sampling_speed.py --preflight-only`, and prints the timing JSON.


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/matyasfarkas/SurrogateNN_DSGE.git"
BRANCH = "codex/colab-jax-gemini-profile"
ROOT = Path("/content/SurrogateNN_DSGE")
OUTPUT = ROOT / "benchmarks" / "results" / "gpuhub_sw07_posterior_ess_calibration_float32_doubling.json"


def run(cmd, cwd=None, check=True, env=None):
    full_env = os.environ.copy()
    if env:
        full_env.update(env)
    print("$", " ".join(map(str, cmd)), flush=True)
    process = subprocess.Popen(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=full_env,
    )
    assert process.stdout is not None
    captured = []
    for line in process.stdout:
        captured.append(line)
        print(line, end="", flush=True)
    returncode = process.wait()
    if check and returncode:
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(map(str, cmd))}"
        )
    return "".join(captured)


print("Python:", sys.version, flush=True)
run(["nvidia-smi"], check=False)

if ROOT.exists():
    run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH])
    run(["git", "-C", str(ROOT), "checkout", BRANCH])
    run(["git", "-C", str(ROOT), "reset", "--hard", f"origin/{BRANCH}"])
else:
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)])

start = time.perf_counter()
run(
    [
        sys.executable,
        str(ROOT / "scripts" / "gpuhub_bootstrap.py"),
        "--mode",
        "calibration",
        "--skip-repo-sync",
        "--root",
        str(ROOT),
        "--preserve-ld-library-path",
        "--preflight-only",
        "--preflight-reps",
        "1",
        "--heartbeat-seconds",
        "15",
        "--dtype",
        "float32",
        "--qme-algorithm",
        "doubling",
    ],
    cwd=ROOT,
)
wall = time.perf_counter() - start
print(f"Total Colab preflight wall seconds: {wall:.3f}", flush=True)

result = json.loads(OUTPUT.read_text())
print("Benchmark:")
print(json.dumps(result["benchmark"], indent=2, sort_keys=True))
print("Runtime summary:")
print(
    json.dumps(
        {
            "jax_version": result["runtime"].get("jax_version"),
            "numpyro_version": result["runtime"].get("numpyro_version"),
            "jax_enable_x64": result["runtime"].get("jax_enable_x64"),
            "jax_default_backend": result["runtime"].get("jax_default_backend"),
            "jax_devices": result["runtime"].get("jax_devices"),
        },
        indent=2,
        sort_keys=True,
    )
)
print("Preflight:")
print(json.dumps(result["preflight"], indent=2, sort_keys=True))
